In [ ]:
from agent_framework import MCPStreamableHTTPTool
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient

In [ ]:
tool = MCPStreamableHTTPTool(
    name="Microsoft Learn MCP",
    url="http://localhost:8000/mcp/",
    # we don't require approval for microsoft_docs_search tool calls
    # but we do for any other tool
    # approval_mode={"never_require_approval": ["microsoft_docs_search"]},
)

In [ ]:
# llm = OpenAIChatClient(
#     api_key="ollama",  # Just a placeholder, Ollama doesn't require API key
#     # base_url="http://localhost:11434/v1",
#     base_url="http://ollama.home/v1",
#     model_id="gpt-oss:20b",
# )
import os
from dotenv import load_dotenv

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
deployment_name = os.getenv("AZURE_OPENAI_MODEL", "")
api_key = os.getenv("AZURE_OPENAI_API_KEY", "")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

llm = AzureOpenAIChatClient(
    endpoint=endpoint or None,
    deployment_name=deployment_name or None,
    api_key=api_key or None,
    api_version=api_version or None,
)

agent = ChatAgent(
    chat_client=llm,
    name="test_agent",
    instructions="You are a helpful agent. You use Model Context Protocol (MCP) tools to answer user questions. "
    "You can only respond using the tools available to you. Do not make up tool functionality. The tools will be"
    "Provided to you in the prompt.",
    tools=[tool],
    temperature=0,
)

query = "what all airlines are in my db?"
print(f"User: {query}")
print("\n=== HTTP Request/Response Details Below ===\n")
result = await agent.run(query)
print("\n=== End of HTTP Details ===\n")
print(f"Agent: {result}\n")

In [ ]:
from random import randint
from typing import Annotated


"""
Ollama with OpenAI Chat Client Example

This sample demonstrates using Ollama models through OpenAI Chat Client by
configuring the base URL to point to your local Ollama server for local AI inference.
Ollama allows you to run large language models locally on your machine.

Environment Variables:
- OLLAMA_ENDPOINT: The base URL for your Ollama server (e.g., "http://localhost:11434/v1/v1/")
- OLLAMA_MODEL: The model name to use (e.g., "mistral", "llama3.2", "phi3")
"""


def get_weather(
    location: Annotated[str, "The location to get the weather for."],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(10, 30)}°C."


agent = ChatAgent(
    chat_client=llm,
    name="weather_agent",
    instructions="You are a helpful agent. You use Model Context Protocol (MCP) tools to answer user questions. "
    "You can only respond using the tools available to you. Do not make up tool functionality. The tools will be"
    "Provided to you in the prompt.",
    tools=[get_weather],
    temperature=0,
)


async def non_streaming_example(agent: ChatAgent) -> None:
    """Example of non-streaming response (get the complete result at once)."""
    print("=== Non-streaming Response Example ===")

    query = "What's the weather like in Seattle?"
    print(f"User: {query}")
    result = await agent.run(query)
    print(f"Agent: {result}\n")


async def streaming_example(agent: ChatAgent, query: str) -> None:
    """Example of streaming response (get results as they are generated)."""
    print("=== Streaming Response Example ===")

    print(f"User: {query}")
    print("Agent: ", end="", flush=True)
    async for chunk in agent.run_stream(query):
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print("\n")

In [ ]:
await streaming_example(agent, "What's the weather like in Portland?")